# Verification — `{{PKG}}` against `{{REVISION}}`

Executed report, not the source of truth. Every mathematical claim lives as a
test under `tests/`; this page runs them and shows the evidence. Existence of
this file proves nothing — `verify` reads whether its cells actually ran.

Re-run headlessly from the repository root:

```
.venv/bin/jupyter nbconvert --to notebook --execute --inplace \
  {{NAME}}/Notebooks/verification.ipynb
```

In [ ]:
import ast
import sys
from pathlib import Path

import numpy as np

ROOT = Path.cwd().parents[1]  # <repo>/{{NAME}}/Notebooks -> <repo>
sys.path.insert(0, str(ROOT / "src"))
rng = np.random.default_rng({{SEED}})

# Which revision each module was written against, read statically.
for file in sorted((ROOT / "src" / "{{PKG}}").rglob("*.py")):
    if file.name == "__init__.py":
        continue
    for node in ast.parse(file.read_text()).body:
        if any(getattr(t, "id", None) == "__provenance__" for t in getattr(node, "targets", [])):
            p = ast.literal_eval(node.value)
            print(f"{file.name:<18} {p['revision']:<26} eqs {','.join(p['equations'])}")
# The fingerprint of the code this report ran against. `execution_count` proves a
# cell ran once; it says nothing about what it ran against, and a report executed
# once and never again stays green while the code moves out from under it. `verify`
# recomputes this digest and compares — so the notebook must not compute one of its
# own: a second formula agrees with the checker only by coincidence, and this one
# did not. `report_digest` is the single implementation, scaffolded into the
# benchmark package, and `src/` is already on the path above.
from {{PKG}}_Benchmark import report_digest

print(report_digest.stamp())


## Levels 1, 2, 4 and 5 — the suite

A red cell here means the implementation does not match what the proposal
claims. Do not edit the assertion to make it pass.

In [ ]:
import pytest

code = pytest.main(["-q", str(ROOT / "tests"), "--rootdir", str(ROOT)])
assert code == 0, f"test suite failed (pytest exit code {code})"

## Level 3 — synthetic evidence

Deterministic, fixed seed, ground truth known by construction. State the
expected behaviour before looking at the output.

In [ ]:
# from {{PKG}}.{{MODULE}} import {{FUNCTION_NAME}}
# Save any figure or table under ../Results/ so the evidence is versioned.
